In [2]:
from Bio import SeqIO
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [8]:
ECOLI_PAIRS_CSV = "ecoli_processed_pairs/ecoli_pairs.csv"
ECOLI_PAIRS_CONCAT_CSV = "ecoli_processed_pairs/ecoli_pairs_concat.csv"

In [9]:
pairs_df = pd.read_csv(ECOLI_PAIRS_CSV, dtype={'genome_id': str})
print(f"Loaded {len(pairs_df)} E. coli pairs from {ECOLI_PAIRS_CSV}")

Loaded 469 E. coli pairs from ecoli_processed_pairs/ecoli_pairs.csv


In [10]:
def load_concat_proteome(faa_path: str) -> str:
    """
    Load multi-FASTA, extract sequences (ignore headers), concat with <PROT> separators.
    Handles your example format.
    """
    proteome = []
    with open(faa_path, 'r') as handle:  # SeqIO.parse works with paths/strings
        for record in SeqIO.parse(handle, 'fasta'):
            seq = str(record.seq).upper()  # AA seq (e.g., MATKTTT....VNG*)
            # Strip trailing * if present (optional, but standard for CDS)
            if seq.endswith('*'):
                seq = seq[:-1]
            proteome.append(f"<PROT>{seq}</PROT>")
    if not proteome:
        raise ValueError(f"No sequences found in {faa_path}")
    return ''.join(proteome)

In [11]:
tqdm.pandas()
pairs_df['mutated_proteome'] = pairs_df['mutated_protein_path'].progress_apply(load_concat_proteome)
pairs_df['unmutated_proteome'] = pairs_df['unmutated_protein_path'].progress_apply(load_concat_proteome)

100%|██████████| 469/469 [00:05<00:00, 78.90it/s]


In [12]:
pairs_df.to_csv('ecoli_processed_pairs/ecoli_pairs_concat.csv', index=False)
print("Updated CSV saved with concatenated proteomes.")

Updated CSV saved with concatenated proteomes.


In [13]:
pairs_df.head()

,genome_id,antibiotic,resistant_phenotype,genome_name,taxon_id,taxon_lineage_names,genome_length,antimicrobial_resistance,antimicrobial_resistance_evidence,mutated_protein_path,unmutated_protein_path,reversions_applied,reversion_count,mutated_proteome,unmutated_proteome
0,562.8523,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5235035,NaN,NaN,ecoli_processed_pairs/562.8523_mutated_protein...,ecoli_processed_pairs/562.8523_unmutated_prote...,True,3,<PROT>MIKFSATLLATLIAASVNAATVDLRIMETTDLHSNMMDFD...,<PROT>MIKFSATLLATLIAASVNAATVDLRIMETTDLHSNMMDFD...
1,562.7542,ciprofloxacin,Resistant,Escherichia coli,562,cellular organisms::Bacteria::Pseudomonadati::...,5589103,NaN,NaN,ecoli_processed_pairs/562.7542_mutated_protein...,ecoli_processed_pairs/562.7542_unmutated_prote...,True,4,<PROT>MATKTTTAPETDSKRTQLFLQSVSIGQNEIPREMIVGCTY...,<PROT>MATKTTTAPETDSKRTQLFLQSVSIGQNEIPREMIVGCTY...
2,562.98417,ciprofloxacin,Resistant,Escherichia coli 99aee782-7bb9-11e9-a8d3-68b59...,562,cellular organisms::Bacteria::Pseudomonadati::...,5031625,Resistant::Susceptible,AMR Panel,ecoli_processed_pairs/562.98417_mutated_protei...,ecoli_processed_pairs/562.98417_unmutated_prot...,True,3,<PROT>MLVAQHTVYFPDAFLTQMREAMPSTLSFDDFLAACQRPLR...,<PROT>MLVAQHTVYFPDAFLTQMREAMPSTLSFDDFLAACQRPLR...
3,562.140842,ciprofloxacin,Resistant,Escherichia coli 018-E1-S06,562,cellular organisms::Bacteria::Pseudomonadati::...,4912847,Resistant::Intermediate::Susceptible,AMR Panel,ecoli_processed_pairs/562.140842_mutated_prote...,ecoli_processed_pairs/562.140842_unmutated_pro...,True,3,<PROT>MRVLLFLLLSLFMLSAFSADNLLRWHDAQHYTVQASTPLK...,<PROT>MRVLLFLLLSLFMLSAFSADNLLRWHDAQHYTVQASTPLK...
4,562.135581,ciprofloxacin,Resistant,Escherichia coli 20200203_MGL_06,562,cellular organisms::Bacteria::Pseudomonadati::...,5058347,NaN,NaN,ecoli_processed_pairs/562.135581_mutated_prote...,ecoli_processed_pairs/562.135581_unmutated_pro...,True,3,<PROT>MLDLFADAEPWQEPLAAGAVILRRFAFNAAEQLIRDINDV...,<PROT>MLDLFADAEPWQEPLAAGAVILRRFAFNAAEQLIRDINDV...
